In [ ]:
# Esame 654AA - a.a. 2025/2026
# Studenti: Leonardo Celati, Samuele Taviano
# Matricole: 660185, ?

In [ ]:
import cup_common as cc
import nn_common as nnc
import torch
import torch.nn as nn
import importlib
from functools import partial
import pandas as pd
import copy
from sklearn.model_selection import KFold, StratifiedKFold
import svm_common as sc

importlib.reload(nnc)
importlib.reload(cc)

<h3>Preparation</h3>
<hr/>

<h4>Data preparation</h4>
<p>The train and test dataset are loaded in form of Pandas DataFrame for better interaction with PyTorch.
Eventually there will these objects:
<ul>
<li>X_tr, y_tr: the train features and train labels (or classes)</li>
<li>X_ts, y_ts: the test features and test labels (or classes)</li>
</ul>
</p>

<h5>Parameters</h5>
<p>Parameters useful for dataset preparation</p>

In [ ]:
# The wine type red,white or join (a common joined red+white dataset)
wine_type="white"
# None or an array with wine quality to exclude from dataset [i.e. 3,8,9]
quality_filter=None

# The seed to be used by data splitter
default_state = 42

# Subset is derived from data analysis
# and feature selection, more for learning purpose
# than for real needs as the number of features is small
subset_features_1 = [
    'alcohol',
    'volatile acidity',
    'sulphates',
    'citric acid',
    'total sulfur dioxide'
]

<h5>Split and prepare</h5>
<p>Dataset is prepared and splitted in two normalized (with MinMaxScaler) datasets: train and test.</p>

In [ ]:
df_train, df_test = cc.load_set()
X_tr, y_tr, X_ts, y_ts = cc.split_and_prepare_dataset(df_train, ratio=0.2)
features_names = X_tr.columns
cc.dataset_introspection(df_train, df_test)

<h4>Network Architecture</h4>
<p>Defining the neural network architecture:
<ul>
<li><b>net</b>: represent the internal NN structure with input, hidden and output layer.
One important thing to consider is that the output layer <u>must be kept linear</u> in order to be able to
plug the loss function and the output adapter</li>
<li><b>output_adapter</b>: the function which takes the output from the NN and apply the latest trasformation (classification, regression, etc...).
Must be coherent with the optimizer_template. See function docs for details</li>
<li><b>model_template</b>: the model itself, a skeleton which holds the net structure, and the output adapter function.
The model is always cloned troughout the notebok, just to keep a template reference.</li>
<li><b>optimizer_template</b>: the weight update algorithm</li>
<li><b>scheduler_template</b>: implements adaptive learning rate decay by using ReduceLROnPlateau together with the optimizer template</li>
</ul>
</p>

In [ ]:
# Starting learning rate
learning_rate = 1e-3

# The input dimension (i.e. the input layer) is obviously
# the number of feature of the TR set
input_dimension = X_tr.shape[1]


# Defining network architecture
# For simplicity we start with just one hidden layer with small nr of hidden unit
hidden_unit = 64
net = nn.Sequential(
    # 1st Hidden Layer
    nn.Linear(input_dimension, hidden_unit, bias=True), # <- NET
    nn.ReLU(), # <- Activation function
    #nn.Tanh(), # <- Activation function

    # Output layer
    # The output layer is kept linear in order to decouple
    # the network architecture from the choice of the loss function
    # and the prediction phase
    nn.Linear(hidden_unit, 4, bias=True) # <- NET
)

# The output and loss function, must be changed according to the type of task
loss_function = nn.MSELoss()
output_adapter = nnc.regression_mse_adapter()

# Model implementation
model_template = nnc.MLP(net, output_adapter)
# weights initialization (for didactic purposes)
model_template.apply(lambda m: nnc.init_weights(m, method="kaiming", nonlinearity="relu"))

# The weight update algorithm
optimizer_template = partial(
    torch.optim.AdamW,
    lr=1e-3,
    weight_decay=1e-4
)

# The factor by learning rate will decrease
learning_rate_decay_factor = 0.5
# The epoch to wait before applying the rate decay factor
learning_rate_decay_patience = 10
# The threshold for measuring the new optimum
learning_rate_decay_threshold = 1e-4

# The learning rate decay function
scheduler_template = partial(
    torch.optim.lr_scheduler.ReduceLROnPlateau,
    mode="min", factor=learning_rate_decay_factor, patience=learning_rate_decay_patience,
    threshold=learning_rate_decay_threshold
)

print(model_template)

In [ ]:


# default_weight_decay = 1e-4
# default_batch_size = 256
#
# #Epochs
# default_epochs=300
# # Early stopping (-1 or < 0 = disabled)
# default_patience=-1
#
default_seed = 1
# default_min_delta = 1e-4
#
# # Custom Hyperparameters
# default_momentum = 0.5
# default_nesterov = True
#
# default_random_state = 42



<h3>NN Run</h3>
<hr/>

<h4>Hold-out</h4>
<p>Training model and prediction according to hold-out strategy.</p>

<h5>Training</h5>
<p>Training the model.</p>

In [ ]:
importlib.reload(nnc)

# custom settings
# Either batch, online or the mini-batch size
ho_batch_size="batch"
ho_epochs=300
# Enabling Early stopping
ho_patience = 50 # patience <0 will disable early stopping
ho_min_delta = 1e-4 # ignored if early stopping disabled

# Clone an untrained model and its optimizer
model_ho = copy.deepcopy(model_template)

train_result = nnc.train(
    model_ho, X_tr, y_tr, optimizer_template, loss_function,
    batch_size=ho_batch_size, epochs=ho_epochs,
    min_delta=ho_min_delta,
    patience=ho_patience,
    seed=default_seed,
    scheduler_template=scheduler_template
)


hist_tr = train_result["hist_tr"]
hist_vl = train_result["hist_vl"]
hist_tr_mee = train_result["hist_tr_mee"]
hist_vl_mee = train_result["hist_vl_mee"]
hist_grad = train_result["hist_grad"]

In [ ]:
importlib.reload(nnc)
nnc.plot_gradient_norm_bars(hist_grad, step=10)

In [ ]:
nnc.plot_epoch_loss(hist_tr, hist_vl)

<h5>MEE</h5>
$$\mathrm{MEE}
=
\frac{1}{N}
\sum_{p=1}^{N}
\left\lVert
\mathbf{o}_p - \mathbf{t}_p
\right\rVert_2
=
\frac{1}{N}
\sum_{p=1}^{N}
\sqrt{
\sum_{k=1}^{4}
(o_{pk} - t_{pk})^2
}$$

In [ ]:
importlib.reload(nnc)
nnc.plot_epoch_mee(hist_tr_mee, hist_vl_mee)

<hr/>

<h4>KFold</h4>
<p>Perform training and validation according to KFold strategy.</p>

<h5>Parameters</h5>
<p>The number of epoch has been increased in respect to hold-out validation, but early stopping have also been enabled to avoid overfitting.</p>
<p>The StratifiedKFold has been choosed as it tryies to keep a balance between classes.</p>

In [ ]:
importlib.reload(nnc)
# StratifiedKFold when shuffle=True shuffle the samples keeping a balance between them
# Passing a random_state permit reproducible output across multiple function calls
kv_random_state=42
kf = KFold(n_splits=5, shuffle=True, random_state=kv_random_state)

# Epochs
kv_epochs=500
kv_batch_size=64
# Disable Early stopping
kv_min_delta = 1e-4
kv_patience = -1

inner_train_params = {
    "epochs": kv_epochs,
    "batch_size": kv_batch_size,
    "seed": default_seed,
}

fold_histories = nnc.run_kfold(
    model_template, X_tr, y_tr, optimizer_template, scheduler_template, loss_function, kf, inner_train_params
)


In [ ]:
nnc.plot_kfold_bar_vl_mee(fold_histories, use="best")

In [ ]:
importlib.reload(nnc)
nnc.plot_kfold_bar_vl_loss(fold_histories, use="best")

In [ ]:
importlib.reload(nnc)
nnc.plot_kfold_bar_vl_rmse(fold_histories, use="best")

In [ ]:
importlib.reload(nnc)
df_kfold = nnc.kfold_regression_table(
    fold_histories,
    use="best",
    derive_rmse=True
)

<h5>Training</h5>
<p>Model is eventually trained with best parameters verified from KFold, in order to predict on official test set.</p>

In [ ]:
# dopo aver scelto best_params dal KFold
model_kf = copy.deepcopy(model_template)

train_kf_result = nnc.train(
    model_kf, X_tr, y_tr, optimizer_template, loss_function,
    epochs=inner_train_params['epochs'], batch_size=inner_train_params['batch_size'],scheduler_template=scheduler_template,
    seed=inner_train_params['seed'], patience=inner_train_params.get('patience',0), min_delta=inner_train_params.get('min_delta',None))

hist_kf_tr = train_kf_result["hist_tr"]
hist_kf_vl = train_kf_result["hist_vl"]
hist_kf_grad = train_kf_result["hist_grad"]

<p>The gradient norm should gradually decrease throughout epochs, indicating that the optimization process moves toward a stable region of the loss landscape.</p>

In [ ]:
nnc.plot_gradient_norm_bars(hist_kf_grad, step=10)

In [ ]:
nnc.plot_epoch_loss(hist_kf_tr, hist_kf_vl)